# FinAgent Pro — Reproducible Experiment

This notebook re-runs a single analysis experiment end-to-end: pulls market data, trains LSTM models (if needed), computes expected returns, performs portfolio optimization, and backtests the resulting portfolio against two baselines: equal-weight and a simple moving-average (MA) filter baseline. It then produces equity curve and drawdown plots and a comparison table of metrics.

In [1]:
# Setup imports and repository path
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
print('Repo root added to path')

Repo root added to path


In [13]:
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from agents.market_agent import run_market_data
try:
    from agents.sentiment_agent import run_sentiment
except ModuleNotFoundError as exc:
    print('Warning: sentiment_agent import failed: %s' % exc)
    def run_sentiment(tickers):
        return {ticker: {'sentiment_score': 0.0} for ticker in tickers}
from agents.risk_agent import run_risk_analysis
from agents.prediction_agent import run_prediction
from agents.portfolio_agent import run_portfolio_optimization

print('Imports OK')

No GPU detected by TensorFlow. (Note: On Windows, TF > 2.10 requires WSL2 for GPU support)
Imports OK


In [14]:
# Experiment configuration
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']
PREDICT_DAYS = 30
EPOCHS = 20
HIST_PERIOD = '1y'
RISK_PROFILE = 'moderate'

print('Tickers:', TICKERS)
print('History period:', HIST_PERIOD)
print('Prediction horizon:', PREDICT_DAYS)
print('Risk profile:', RISK_PROFILE)

Tickers: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']
History period: 1y
Prediction horizon: 30
Risk profile: moderate


In [15]:
# Helper functions

def compute_portfolio_backtest(tickers, allocation, period='1y'):
    import yfinance as yf
    import pandas as pd
    import numpy as np

    if len(tickers) == 0:
        return None

    df = pd.DataFrame()
    for t in tickers:
        try:
            prices = yf.download(t, period=period, progress=False)['Close'].rename(t)
        except Exception as exc:
            print(f'Backtest download failed for {t}:', exc)
            return None
        df = pd.concat([df, prices], axis=1)

    df = df.dropna()
    if df.empty:
        return None

    returns = df.pct_change().dropna()
    weights = pd.Series({t: allocation.get(t, 0) for t in returns.columns})
    weights = weights.reindex(returns.columns).fillna(0)
    weights = weights / weights.sum() if weights.sum() != 0 else weights

    port_daily = returns.dot(weights)
    cumulative = (1 + port_daily).cumprod()

    total_return = float(cumulative.iloc[-1] - 1)
    annual_return = float(port_daily.mean() * 252)
    annual_volatility = float(port_daily.std() * np.sqrt(252))
    sharpe = float((annual_return - 0.05) / annual_volatility) if annual_volatility != 0 else None
    roll_max = cumulative.cummax()
    drawdown = (cumulative / roll_max) - 1
    max_dd = float(drawdown.min())

    return {
        'total_return': total_return,
        'annual_return': annual_return,
        'annual_volatility': annual_volatility,
        'sharpe': sharpe,
        'max_drawdown': max_dd,
        'equity_curve': cumulative,
        'daily_returns': port_daily
    }

def equal_weight_allocation(tickers):
    if len(tickers) == 0:
        return {}
    weight = 1.0 / len(tickers)
    return {t: weight for t in tickers}

def ma_filter_allocation(tickers, period='1y', sma_window=50):
    import yfinance as yf
    import pandas as pd

    df = pd.DataFrame()
    for t in tickers:
        try:
            prices = yf.download(t, period=period, progress=False)['Close'].rename(t)
        except Exception as exc:
            print(f'MA filter download failed for {t}:', exc)
            return equal_weight_allocation(tickers)
        df = pd.concat([df, prices], axis=1)

    df = df.dropna()
    if df.empty:
        return equal_weight_allocation(tickers)

    sma = df.rolling(window=sma_window).mean().iloc[-1]
    current = df.iloc[-1]
    selected = [t for t in tickers if current[t] > sma[t]]
    if len(selected) == 0:
        return equal_weight_allocation(tickers)

    weight = 1.0 / len(selected)
    return {t: weight if t in selected else 0.0 for t in tickers}

def plot_equity_curves(curves, title='Equity Curves'):
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12, 6))
    for label, curve in curves.items():
        normalized = curve / curve.iloc[0]
        plt.plot(normalized.index, normalized.values, label=label)
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('Normalized Equity')
    plt.legend()
    plt.grid(True)
    plt.show()

def print_metrics_table(metrics_dict):
    import pandas as pd
    df = pd.DataFrame(metrics_dict).T
    display(df.round(4))


In [16]:
# Run agents and collect data
try:
    market_data = run_market_data(TICKERS, period=HIST_PERIOD)
except Exception as exc:
    market_data = {}
    print('Market agent failed:', exc)
try:
    sentiment_data = run_sentiment(TICKERS)
except Exception as exc:
    sentiment_data = {}
    print('Sentiment agent failed:', exc)
try:
    risk_metrics = run_risk_analysis(TICKERS, period=HIST_PERIOD)
except Exception as exc:
    print('Risk agent failed:', exc)
    risk_metrics = {ticker: {'variance': 0.04, 'beta': 1.0, 'sharpe_ratio': 0.0} for ticker in TICKERS}
try:
    prediction_data = run_prediction(TICKERS, predict_days=PREDICT_DAYS, epochs=EPOCHS)
except Exception as exc:
    print('Prediction agent failed:', exc)
    prediction_data = {ticker: {'expected_return': 0.05, 'backtest_mse': None, 'model_path': None} for ticker in TICKERS}
expected_returns = {ticker: prediction_data.get(ticker, {}).get('expected_return', 0.0) for ticker in TICKERS}
print('Expected returns (annualized)')
for ticker, exp_return in expected_returns.items():
    print(f'{ticker}: {exp_return:.4f}')


Error fetching data for AAPL: Too Many Requests. Rate limited. Try after a while.
Error fetching data for MSFT: Too Many Requests. Rate limited. Try after a while.
Error fetching data for GOOGL: Too Many Requests. Rate limited. Try after a while.
Error fetching data for AMZN: Too Many Requests. Rate limited. Try after a while.
Error fetching data for NVDA: Too Many Requests. Rate limited. Try after a while.
Risk agent failed: Too Many Requests. Rate limited. Try after a while.
Starting prediction for AAPL...
YF.download() has changed argument auto_adjust default to True



1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Insufficient data for AAPL, using fallback.
Starting prediction for MSFT...



1 Failed download:
['MSFT']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Insufficient data for MSFT, using fallback.
Starting prediction for GOOGL...


...

Insufficient data for GOOGL, using fallback.
Starting prediction for AMZN...



1 Failed download:
['AMZN']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Insufficient data for AMZN, using fallback.
Starting prediction for NVDA...



1 Failed download:
['NVDA']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Insufficient data for NVDA, using fallback.
Expected returns (annualized)
AAPL: 0.0500
MSFT: 0.0500
GOOGL: 0.0500
AMZN: 0.0500
NVDA: 0.0500


In [17]:
# Build and optimize allocation
allocation = run_portfolio_optimization(TICKERS, expected_returns, risk_metrics, risk_profile=RISK_PROFILE, history_period=HIST_PERIOD)
allocation = {t: w for t, w in allocation.items() if w >= 0.01}
print('Optimized allocation:')
print(allocation)

# Baseline allocations
equal_alloc = equal_weight_allocation(TICKERS)
ma_alloc = ma_filter_allocation(TICKERS, period=HIST_PERIOD)
print('\nEqual-weight allocation:')
print(equal_alloc)
print('\nMA-filter allocation:')
print(ma_alloc)

Optimized allocation:
{'AAPL': 0.2, 'MSFT': 0.2, 'GOOGL': 0.2, 'AMZN': 0.2, 'NVDA': 0.2}



1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['MSFT']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['GOOGL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['AMZN']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['NVDA']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')



Equal-weight allocation:
{'AAPL': 0.2, 'MSFT': 0.2, 'GOOGL': 0.2, 'AMZN': 0.2, 'NVDA': 0.2}

MA-filter allocation:
{'AAPL': 0.2, 'MSFT': 0.2, 'GOOGL': 0.2, 'AMZN': 0.2, 'NVDA': 0.2}


In [18]:
# Backtest the allocations
results = {}
results['Optimized'] = compute_portfolio_backtest(list(allocation.keys()), allocation, period=HIST_PERIOD)
results['Equal Weight'] = compute_portfolio_backtest(TICKERS, equal_alloc, period=HIST_PERIOD)
results['MA Filter'] = compute_portfolio_backtest(TICKERS, ma_alloc, period=HIST_PERIOD)

summary = {}
curves = {}
for label, result in results.items():
    if result is None:
        continue
    summary[label] = {
        'total_return': result['total_return'],
        'annual_return': result['annual_return'],
        'annual_volatility': result['annual_volatility'],
        'sharpe': result['sharpe'],
        'max_drawdown': result['max_drawdown']
    }
    curves[label] = result['equity_curve']

print('Backtest summary:')
print_metrics_table(summary)


1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['MSFT']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['GOOGL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['AMZN']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['NVDA']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['MSFT']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['GOOGL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['AMZN']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')

1 Failed download:
['NVDA']: YFRateLimitError('Too Many Requests. Rate limited.

Backtest summary:


""


In [19]:
# Plot equity curves for each allocation
if not curves:
    print('No equity curves available to plot')
else:
    plot_equity_curves(curves, title='Portfolio Equity Curves')


No equity curves available to plot


## Notes

- This notebook uses the existing repo agents and portfolio optimization functions.
- If the sentiment agent fails due to invalid API credentials, the notebook continues.
- The portfolio backtest uses historical daily closes over the selected period.